In [2]:
import os
import re
import pandas as pd

# 1. Load Excel file and automatically locate the July Sheet
file_path = "Mintt_CLC Live.xlsx"
xls = pd.ExcelFile(file_path)

# Target sheet containing 'july' in its name
july_sheet = next(
    (sheet for sheet in xls.sheet_names if "july" in sheet.lower()), None
)

if not july_sheet:
    raise ValueError("Could not find a July sheet in the workbook.")

df = pd.read_excel(file_path, sheet_name=july_sheet)

# 2. Clean dataset (remove empty divider rows)
valid_mask = (
    df.iloc[:, 1].notna() | df.iloc[:, 3].notna() | df.iloc[:, 4].notna()
)
df = df[valid_mask].copy()

# Combine text for row-level searching
df["Full_Text"] = df.astype(str).apply(lambda row: " ".join(row), axis=1)

# Forward-fill date column down multi-line lead records
df["Parsed_Date"] = pd.to_datetime(df.iloc[:, 1], errors="coerce").ffill()


# 3. Categorize into weekly ranges
def assign_week(dt):
    if pd.isna(dt):
        return "Unknown Date"
    if dt.month == 7:
        if dt.day <= 10:
            return "Week 1 (July 1 - July 10)"
        elif dt.day <= 20:
            return "Week 2 (July 11 - July 20)"
        else:
            return "Week 3 (July 21 - July 31)"
    elif dt.month == 8:
        return "Week 4 (August 1 - August 7)"
    return f"Month {dt.month}"


df["Assigned_Week"] = df["Parsed_Date"].apply(assign_week)

# 4. Define Search Metrics
df["Ringing"] = df["Full_Text"].str.contains("Ringing", case=False, na=False)

interested_pat = (
    r"consultation done|pgs|schedule|interested|will discuss|call back"
)
df["Interested"] = df["Full_Text"].str.contains(
    interested_pat, case=False, na=False
)

potential_pat = (
    r"will inform|schedule|discuss with husband|callback|ready for consultation"
)
df["Potential"] = df["Full_Text"].str.contains(
    potential_pat, case=False, na=False
) & ~df["Full_Text"].str.contains(
    "not interested|cancelled|decline", case=False, na=False
)

not_conv_pat = r"cancelled|declined|not picking|does not receive|not interested|cannot afford|distance issue"
df["Interested_Not_Converted"] = (
    df["Interested"]
    & df["Full_Text"].str.contains(not_conv_pat, case=False, na=False)
    & ~df["Full_Text"].str.contains("consultation done", case=False, na=False)
)

reasons = {
    "Distance / Location Issue": r"distance|location|far|cannot travel",
    "Not Comfortable with Online": r"online|not comfotable with online|hectic",
    "No Concern / Misdial": r"does not have any concern|no concern|test lead",
    "High Fees / Budget Constraint": r"cannot afford|fees|fee|budget",
    "Cancelled / Did Not Attend": r"cancelled|cancelled last moment|did not answer",
    "Child Health / Exam Delay": r"exam|health issue|younger daughter",
    "Taking Therapy Elsewhere / Local": r"in our city|other center|already taking|local",
}

for col_name, pattern in reasons.items():
    df[col_name] = df["Full_Text"].str.contains(pattern, case=False, na=False)

# 5. Build Aggregated Summary Table
weeks_order = [
    "Week 1 (July 1 - July 10)",
    "Week 2 (July 11 - July 20)",
    "Week 3 (July 21 - July 31)",
    "Week 4 (August 1 - August 7)",
]

summary_rows = []
for w in weeks_order:
    w_df = df[df["Assigned_Week"] == w]
    row_data = {
        "Week": w,
        "Total Leads": len(w_df),
        "Ringing Customers": w_df["Ringing"].sum(),
        "Interested Clients": w_df["Interested"].sum(),
        "Potential Clients": w_df["Potential"].sum(),
        "Interested but Not Converted": w_df["Interested_Not_Converted"].sum(),
    }
    for r_name in reasons.keys():
        row_data[r_name] = w_df[r_name].sum()
    summary_rows.append(row_data)

summary_df = pd.DataFrame(summary_rows)

# 6. Save Multi-Sheet Excel Workbook
output_filename = "July_2026_Weekly_Report.xlsx"
with pd.ExcelWriter(output_filename, engine="openpyxl") as writer:
    summary_df.to_excel(
        writer, sheet_name="Executive_Summary", index=False
    )
    df.to_excel(writer, sheet_name="July_Lead_Audit_Trail", index=False)

print(
    f"Report successfully created and exported to '{output_filename}'!\n"
)
print(summary_df.to_string(index=False))

Report successfully created and exported to 'July_2026_Weekly_Report.xlsx'!

                        Week  Total Leads  Ringing Customers  Interested Clients  Potential Clients  Interested but Not Converted  Distance / Location Issue  Not Comfortable with Online  No Concern / Misdial  High Fees / Budget Constraint  Cancelled / Did Not Attend  Child Health / Exam Delay  Taking Therapy Elsewhere / Local
   Week 1 (July 1 - July 10)           28                 18                  11                  8                             1                         20                            2                     2                              1                           0                          1                                 1
  Week 2 (July 11 - July 20)           34                 23                  13                  5                             6                         19                            1                     0                              3                           3 